# 03 · L'évaluation honnête sur le test scellé (PF.8, et l'audit de PF.9)

Le livrable de l'étape PF.8 : ce notebook **exécuté une fois**, `experiments/test_results.json` et la section « Les résultats » de `docs/rapport.md`. Le test s'ouvre **une seule fois**, pour le modèle retenu et les deux baselines, après la fin des expériences : `evaluate.open_sealed_test` vérifie le découpage par son SHA-256 (celui de `docs/protocole.md`), refuse une seconde ouverture et note celle-ci dans `experiments/test_access.jsonl`.

Chaque étape tourne dans `with step(...)` : tant qu'une fonction de ton paquet lève `NotImplementedError`, ou qu'une étape précédente n'a pas tourné, la cellule affiche ⏳ au lieu d'une erreur, et « Tout exécuter » va jusqu'au bout. Le notebook n'a pas de code d'apprentissage à lui : il appelle ton paquet (`src/projet/`). Ajoute tes propres cellules (figures, vérifications) là où le disent les `TODO`.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True   # True: configs/fast.yaml (CPU, < 10 min); False: configs/full.yaml (🚀 GPU Colab advised)
REPO_ON_DRIVE = "/content/drive/MyDrive/TODO-your-repository"   # on Colab only: where your repository is cloned

import os
import sys
from contextlib import contextmanager
from pathlib import Path

import pandas as pd

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    os.chdir(REPO_ON_DRIVE)

ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "fast.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "configs" / "fast.yaml").exists():
    raise FileNotFoundError("Projet introuvable : ouvre ce notebook depuis ton dépôt (celui qui a configs/fast.yaml).")
os.chdir(ROOT)                                    # the paths of the configuration are relative to the project
(ROOT / "figures").mkdir(exist_ok=True)           # the figures of the report: plt.savefig("figures/....png")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))         # your package, even without pip install -e


class Pending(Exception):
    """An earlier step of the notebook did not run."""


def need(**values):
    """Raise Pending if one of the values is still None (the step that computes it is not done)."""
    missing = [name for name, value in values.items() if value is None]
    if missing:
        raise Pending("attends d'abord : " + ", ".join(missing))


@contextmanager
def step(title):
    """Run one step: a function not written yet, or a missing earlier step, prints ⏳ instead of an error."""
    try:
        yield
    except NotImplementedError as err:
        print(f"⏳ {title} : pas encore écrit ({err})")
    except Pending as err:
        print(f"⏳ {title} : {err}")


from projet.config import default_config_path, load_config  # noqa: E402

CONFIG_PATH = default_config_path(FAST_MODE)
config = load_config(CONFIG_PATH)
print(f"Projet : {ROOT.name} · mode {'rapide' if FAST_MODE else 'complet'} ({CONFIG_PATH.name}) · tâche : {config.task}")

## 0. Avant d'ouvrir le test

**La répétition générale.** Tant que `OPEN_TEST = False`, tout le notebook tourne sur la **validation** à la place du test : chaque cellule, les tiennes comprises (robustesse, explications), s'essaie sans toucher au test, et rien n'est écrit dans `experiments/`. Le test ne s'ouvre qu'une fois : une cellule qui plante après son ouverture ne se rattrape qu'en le rouvrant, et la réouverture se rapporte.

Coche tout avant de passer `OPEN_TEST = True` :

- « Tout exécuter » en répétition (`OPEN_TEST = False`) va jusqu'au bout, sans ⏳ ni erreur, sur la validation ;
- le modèle retenu est figé : son artefact est dans `config.out_dir` (`projet train --config configs/full.yaml`) et son run dans le journal ;
- les choix sont faits sur la validation (hyperparamètres, seuil, epoch d'arrêt), les ablations aussi ;
- `docs/protocole.md` est tagué `protocole`, et toute modification depuis le tag est écrite dans sa section « Modifications » ;
- tout est commité (`git status` est vide).

Après l'exécution : `open_sealed_test` a déjà commité sa ligne de `experiments/test_access.jsonl` ; commite **aussitôt** `experiments/test_results.json` et `experiments/test_predictions.csv` (une prédiction par exemple du test, pour les trois modèles : une analyse faite plus tard les relit sans rouvrir le test), puis le notebook exécuté. Le relancer ensuite (un relecteur, « Tout exécuter » dans un clone) ne rouvre pas le test : la cellule d'ouverture voit la ligne du journal, le dit, et les cellules suivantes affichent ⏳ ; les sorties qui comptent sont celles du notebook commité. Pour refaire une évaluation interrompue ou fausse, `ALLOW_REOPEN = True` : la réouverture est notée dans le journal du test, et tu la rapportes, avec les deux résultats.

In [ ]:
OPEN_TEST = False      # True ONCE, when the model is final (the checklist above)
ALLOW_REOPEN = False   # True only to redo an interrupted or wrong evaluation: it is logged, say it in the report
REASON = "TODO: final evaluation of the model of run <run_id>"
DELIVERED = config.out_dir   # the artifact of the retained run; say in the protocol which seed it is (chosen before
                             # the seeds are compared) and on which rows it was trained

In [ ]:
with step("état du protocole (evaluate.protocol_status)"):
    from projet.evaluate import protocol_status, sealed_test_accesses

    status = protocol_status()
    print("tag « protocole » :", status["tag_commit"] or "⚠️ absent")
    print("protocole modifié depuis le tag :", status["protocol_changed_since_tag"])
    print("modifications non commitées du protocole ou du découpage :", status["uncommitted"])
    print("ouvertures précédentes du test :", len(sealed_test_accesses(config)))

## 1. Les trois modèles

Le modèle retenu, rechargé depuis son artefact (`predict.load_artifact`) comme le fera l'API, et les deux baselines, réentraînées sur **les mêmes exemples d'entraînement que lui** (le sous-échantillon du mode rapide compris) avec leurs réglages de PF.5.

In [ ]:
predictor = naive = classical = None
with step("le modèle retenu (predict.load_artifact)"):
    from projet.predict import load_artifact

    if not Path(DELIVERED).exists():
        raise Pending(f"l'artefact {DELIVERED} (projet train --config {CONFIG_PATH.as_posix()})")
    predictor = load_artifact(DELIVERED)
with step("les deux baselines, réentraînées"):
    from projet.data import load_examples, load_splits, select, subsample
    from projet.models import build_classical, build_naive

    examples = load_examples(config)
    # the same training rows as the deep model (cli.py): the fast mode subsamples them for both
    fit_df = subsample(select(examples, load_splits(config.split_file), "train"), config.max_train_examples, config.seed)
    # TODO PF.8: pd.concat with the validation split if your protocol refits the baselines on train + val
    naive = build_naive(config).fit(fit_df, fit_df[config.target])
    classical = build_classical(config).fit(fit_df, fit_df[config.target])

## 2. L'ouverture du test (ou la répétition sur la validation)

Avec `OPEN_TEST = False`, `eval_df` est la validation : la répétition générale. Avec `OPEN_TEST = True`, c'est le test, ouvert une fois, et seulement si les trois modèles sont prêts à prédire.

In [ ]:
eval_df = None
EVAL_SPLIT = "test" if OPEN_TEST else "val"
if not OPEN_TEST:
    with step("répétition générale sur la validation (le test reste scellé)"):
        need(predictor=predictor, naive=naive, classical=classical)
        eval_df = select(examples, load_splits(config.split_file), "val")
        print(f"ℹ️ Le test reste scellé (OPEN_TEST = False) : la suite tourne sur la validation ({len(eval_df)} "
              "exemples), pour répéter l'évaluation. Rien n'est écrit dans experiments/.")
else:
    with step("ouvrir le test scellé (evaluate.open_sealed_test)"):
        from projet.evaluate import open_sealed_test, sealed_test_accesses

        need(predictor=predictor, naive=naive, classical=classical)   # never spend the one opening on a model not ready
        previous = sealed_test_accesses(config)
        if previous and not ALLOW_REOPEN:
            print(f"🔒 Le test a déjà été ouvert ({len(previous)} fois, la première le {previous[0].get('opened')}) : "
                  "cette exécution ne le rouvre pas. Les résultats de l'ouverture sont dans "
                  "experiments/test_results.json et dans les sorties du notebook commité ; pour refaire une "
                  "évaluation interrompue ou fausse : ALLOW_REOPEN = True, et dis-le dans le rapport.")
        else:
            eval_df = open_sealed_test(config, reason=REASON, allow_reopen=ALLOW_REOPEN)
            print(f"🔓 test ouvert : {len(eval_df)} exemples (l'ouverture est notée et commitée).")

## 3. Les métriques, avec leurs intervalles

Pour chaque modèle : les métriques du protocole (`evaluate.compute_metrics`, avec les probabilités de toutes les classes pour ROC-AUC et PR-AUC ; une métrique principale de probabilités se bootstrape sur elles) et l'intervalle bootstrap à 95 % de la métrique principale (`evaluate.bootstrap_ci`, par groupes si le découpage en a) ; puis la différence entre le modèle retenu et chaque baseline, sur les **mêmes** tirages (`evaluate.paired_bootstrap`). Un chiffre sans intervalle ni baseline ne se rapporte pas.

In [ ]:
import json

import numpy as np

predictions = scores = results = None
with step("prédictions des trois modèles"):
    need(eval_df=eval_df, predictor=predictor, naive=naive, classical=classical)
    y_true = eval_df[config.target].to_numpy()
    out = predictor.predict(eval_df.drop(columns=[config.target]).to_dict("records"))
    predictions = {"naive": np.asarray(naive.predict(eval_df)), "classical": np.asarray(classical.predict(eval_df)),
                   "model": np.array([o.get("label", o.get("prediction")) for o in out])}
    confidence = np.array([o.get("score", np.nan) for o in out], dtype=float)   # of the PREDICTED class: sorts errors
    if config.task == "classification" and all("proba" in o for o in out):
        classes = list(classical.classes_)                  # the order of the columns of predict_proba
        scores = {"naive": naive.predict_proba(eval_df), "classical": classical.predict_proba(eval_df),
                  "model": np.array([[o["proba"][str(c)] for c in classes] for o in out])}
        if len(classes) == 2:                                # binary: the probability of the positive class
            scores = {name: s[:, 1] for name, s in scores.items()}
with step("métriques et intervalles bootstrap"):
    need(predictions=predictions)
    from projet.evaluate import bootstrap_ci, compute_metrics, paired_bootstrap

    # a main metric of probabilities is bootstrapped on the scores, any other on the labels
    USES_SCORES = config.metric.startswith(("roc_auc", "average_precision", "neg_log_loss", "neg_brier_score"))
    if USES_SCORES:
        need(scores=scores)                                  # Predictor.predict must give "proba"

    def main_metric(y, p):
        if USES_SCORES:
            return compute_metrics(y, None, config, y_score=p)[config.metric]
        return compute_metrics(y, p, config)[config.metric]

    measured = scores if USES_SCORES else predictions

    groups = eval_df[config.group_column].to_numpy() if config.group_column else None
    boot = {"n_boot": config.n_bootstrap, "seed": config.seed, "groups": groups}
    results = {"split": EVAL_SPLIT, "metric": config.metric, "n_examples": int(len(eval_df)), "models": {},
               "differences": {}}
    for name, p in predictions.items():
        low, high = bootstrap_ci(y_true, measured[name], main_metric, **boot)
        y_score = None if scores is None else scores[name]
        results["models"][name] = {**compute_metrics(y_true, p, config, y_score=y_score), "ci95_low": low,
                                   "ci95_high": high}
    for base in ("naive", "classical"):
        results["differences"][f"model - {base}"] = paired_bootstrap(y_true, measured["model"], measured[base],
                                                                      main_metric, **boot)
    display(pd.DataFrame(results["models"]).T.round(4))
    display(pd.DataFrame(results["differences"]).T.round(4))

In [ ]:
with step("écrire experiments/test_results.json"):
    need(results=results)
    if not OPEN_TEST:
        raise Pending("la répétition sur la validation n'écrit rien ; le fichier s'écrit à l'ouverture du test")
    path = Path("experiments/test_results.json")
    if path.exists():                    # never overwrite the first evaluation
        path = path.with_name(f"test_results_reopened_{len(list(path.parent.glob('test_results*.json')))}.json")
        print(f"⚠️ le test a déjà été évalué : ce résultat va dans {path.name}, à signaler dans le rapport")
    path.write_text(json.dumps(results, indent=1, default=float) + "\n", encoding="utf-8")
    # one row per test example: its id, the truth, the three predictions (and scores): no input, so reading it later
    # is not a new opening (error analysis, subgroups computed afterwards from the ids)
    table = pd.DataFrame({"id": eval_df["id"].astype(str).to_numpy(), "y_true": y_true,
                          **{f"pred_{name}": p for name, p in predictions.items()}})
    if scores is not None and all(np.ndim(s) == 1 for s in scores.values()):
        table = table.assign(**{f"score_{name}": s for name, s in scores.items()})
    table_path = path.with_name(path.name.replace("test_results", "test_predictions").replace(".json", ".csv"))
    table.to_csv(table_path, index=False)
    print(f"✅ {path} et {table_path} écrits : commite-les maintenant, avec le notebook exécuté")

## 4. L'analyse d'erreurs

Les erreurs les plus sûres d'elles (`evaluate.worst_errors`), à lire **une par une** et à ranger par cause : label douteux, cas ambigu, sous-groupe rare, défaut de prétraitement, vrai échec du modèle. Garde trois exemples parlants pour le rapport et la soutenance.

In [ ]:
with step("les erreurs les plus sûres d'elles (evaluate.worst_errors)"):
    need(predictions=predictions)
    from projet.evaluate import worst_errors

    errors = worst_errors(eval_df, y_true, predictions["model"], y_score=confidence, k=20)
    with pd.option_context("display.max_columns", None, "display.width", 250):
        display(errors)                                       # every column: the ones that explain an error too
    # TODO PF.8: your reading of the errors, grouped by cause (in the report). Tabular data: read an error by its
    # profile, e.g. errors[features] - train_rows.groupby(config.target)[features].median() for its true class
    # (train_rows: select(examples, splits, "train")), and look at the features that pull it away

## 5. Les sous-groupes et la robustesse

Les sous-groupes décidés dans le protocole (classes, sources, périodes, et une variable sensible si tu as le droit de t'en servir), chacun avec son effectif et son intervalle (`evaluate.subgroup_table`) ; puis les perturbations de robustesse du protocole (bruit, décalage, autre période). Un sous-groupe de moins de `min_size` exemples se signale, il ne se conclut pas.

In [ ]:
SUBGROUP_COLUMN = None   # TODO PF.8: the column of eval_df that holds the subgroups of your protocol
with step("métrique par sous-groupe (evaluate.subgroup_table)"):
    need(predictions=predictions, SUBGROUP_COLUMN=SUBGROUP_COLUMN)
    from projet.evaluate import subgroup_table

    display(subgroup_table(y_true, predictions["model"], eval_df[SUBGROUP_COLUMN].to_numpy(), main_metric,
                           n_boot=config.n_bootstrap, seed=config.seed))
with step("robustesse"):
    need(predictions=predictions)
    # TODO PF.8: the perturbations of your protocol (noise, shift, another period): predict, compute, compare
    raise NotImplementedError("TODO PF.8: robustness")

## 6. Explicabilité et équité (PF.9)

Des attributions adaptées au modèle (B6) : SHAP ou importance par permutation pour un modèle tabulaire (B6.12, B6.15), Grad-CAM ou gradients intégrés pour des images (B6.18 à B6.20), attributions par token pour du texte ; avec leur **test de santé** (B6.22 : une explication qui ne change pas quand on brouille les poids n'explique rien). Puis l'audit par sous-groupe (B6.23, B6.24) avec ses intervalles, et ce qu'il ne prouve pas.

In [ ]:
with step("explications du modèle retenu"):
    need(predictions=predictions)
    # TODO PF.9: attributions on a few test examples (good predictions AND errors), their sanity check, the figures
    raise NotImplementedError("TODO PF.9: attributions")

## Ce que tu retiens

Recopie dans `docs/rapport.md` (« Les résultats », « Explicabilité et équité ») et dans `model_card.md` : la date et le commit de l'ouverture du test (`experiments/test_access.jsonl`), le tableau des trois modèles avec leurs intervalles, les différences appariées, le critère de succès **atteint ou non** (un échec honnête vaut mieux qu'un succès douteux), l'analyse d'erreurs par cause, les sous-groupes avec leurs effectifs, la robustesse, les explications et leurs limites.